In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train BEV v4 Cleaned + DINOv2 ViT-Base

Self-contained notebook for `v4` training with:
- merged `train + val` pool;
- empty-GT removal;
- smart deduplication by near-static consecutive frames;
- test-matched validation split targeting about 200 samples;
- rover embeddings with `Other=0`, top-25 frequent train rovers getting unique IDs;
- `DINOv2 ViT-Base` image backbone loaded through `torch.hub`;
- safe train loop with only `val@0.5` and `ema val@0.5` during training.

This notebook does not modify the original dataset structure. All caches, hub downloads, and checkpoints are written only to `RUN_DIR`.



In [1]:
!ls /kaggle

huggingface  input  lib  nbdev	src  working


In [2]:
import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_train_kaggle_safe_v2/autonomy_yandex_dataset_train/')
DATA_VAL   = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_val_kaggle_safe/autonomy_yandex_dataset_val/')
DATA_TEST  = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_test_kaggle_safe/autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_dinov2_cleaned',
    'img_hw': (392, 700),
    'batch_size': 4,
    'val_batch_size': 1,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'freeze_backbone_epochs': 2,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'backbone_name': 'dinov2_vitb14',
    'hub_repo': 'facebookresearch/dinov2',
    'backbone_out_dim': 768,
    'backbone_proj_dim': 128,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TORCH_HUB_DIR = RUN_DIR / 'torch_hub'
TORCH_HUB_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(TORCH_HUB_DIR))

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('device:', device)
if device.type == 'cuda':
    print('cuda devices:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
print('torch hub dir:', TORCH_HUB_DIR)
print('img_hw:', cfg['img_hw'])
print('train batch_size:', cfg['batch_size'], '| train workers:', cfg['num_workers'])
print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)



device: cuda
cuda devices: 2
0 Tesla T4
1 Tesla T4
torch hub dir: runs/v4_dinov2_cleaned/torch_hub
img_hw: (392, 700)
train batch_size: 4 | train workers: 4
val batch_size: 1 | val workers: 0


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [6]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))


Smart dedup groups: 100%|██████████| 1473/1473 [02:31<00:00,  9.72it/s]


{
  "merged_before_clean": 5000,
  "removed_empty_gt": 117,
  "after_empty_filter": 4883,
  "removed_by_dedup": 390,
  "clean_total": 4493,
  "dedup_groups": 173
}


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,...,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order,__data_root,__source_split,coverage,valid_count,pos_count
0,autonomy_yandex_dataset_train/static_grids/163...,1633857774533809000,12:18:58,1633857774533809000,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,...,autonomy_yandex_dataset_train/predicted_static...,2021-10-10,12:18:58,orvy,0,/kaggle/input/datasets/letiti6e/bev-yandex/aut...,train,0.061972,1468,602
1,autonomy_yandex_dataset_train/static_grids/163...,1636812143899937000,15:34:56,1636812143899937000,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,...,autonomy_yandex_dataset_train/predicted_static...,2021-11-13,15:34:56,shelly,0,/kaggle/input/datasets/letiti6e/bev-yandex/aut...,train,0.227077,5379,3752
2,autonomy_yandex_dataset_train/static_grids/163...,1633600207233930000,12:28:29,1633600207233930000,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,...,autonomy_yandex_dataset_train/predicted_static...,2021-10-07,12:28:29,orvy,0,/kaggle/input/datasets/letiti6e/bev-yandex/aut...,train,0.225008,5330,1264


,kept_row_id,group_size,members
0,2271,2,"[2271, 3683]"
1,4101,2,"[4473, 4101]"
2,1366,3,"[306, 2710, 1366]"
3,98,5,"[1721, 2885, 209, 3630, 98]"
4,4151,2,"[4824, 4151]"
5,1703,3,"[526, 277, 1703]"
6,4867,3,"[4332, 4815, 4867]"
7,2549,2,"[3137, 2549]"
8,2112,2,"[2112, 3216]"
9,2330,2,"[2938, 2330]"


train_idx: 4273 val_idx: 220
num rover classes including Other: 26
top rover mapping: {'__other__': 0, 'orvy': 1, 'shelly': 2, 'lerita': 3, 'ward': 4, 'ravine': 5, 'greben': 6, 'lucky': 7, 'miro': 8, 'benzon': 9, 'natelio': 10, 'darron': 11, 'greton': 12, 'jurgie': 13, 'onipa': 14, 'targi': 15, 'kynde': 16, 'soan': 17, 'baland': 18, 'mika': 19, 'crozby': 20, 'litov': 21, 'hetera': 22, 'hankie': 23, 'stelard': 24, 'nikena': 25}


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.decoder import SmallUNet, _UNetBlock

class _DINOv2ViTBackbone(nn.Module):
    def __init__(self,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 out_dim: int = 768,
                 proj_dim: int = 128,
                 patch_size: int = 14):
        super().__init__()
        self.hub_repo = hub_repo
        self.backbone_name = backbone_name
        self.out_dim = out_dim
        self.patch_size = patch_size
        self.vit = self._load_hub_model()
        self.proj = nn.Conv2d(out_dim, proj_dim, 1)

    def _load_hub_model(self):
        last_err = None
        attempts = [
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, pretrained=True),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github'),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github', pretrained=True),
        ]
        for kwargs in attempts:
            try:
                return torch.hub.load(**kwargs)
            except Exception as e:
                last_err = e
        raise RuntimeError(
            f'Failed to load DINOv2 backbone {self.backbone_name} from {self.hub_repo}. '
            f'Last error: {last_err}'
        )

    def _extract_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        B, _, H, W = x.shape
        Hp = H // self.patch_size
        Wp = W // self.patch_size
        expected_tokens = Hp * Wp

        if hasattr(self.vit, 'get_intermediate_layers'):
            try:
                inter = self.vit.get_intermediate_layers(x, n=1, reshape=True, return_class_token=False)
                feat = inter[0] if isinstance(inter, (list, tuple)) else inter
                if isinstance(feat, (list, tuple)):
                    feat = feat[0]
                if feat.ndim == 4:
                    return feat
            except Exception:
                pass

        tokens = None
        if hasattr(self.vit, 'forward_features'):
            feats = self.vit.forward_features(x)
            if isinstance(feats, dict):
                if 'x_norm_patchtokens' in feats:
                    tokens = feats['x_norm_patchtokens']
                elif 'patch_tokens' in feats:
                    tokens = feats['patch_tokens']
                elif 'x_prenorm' in feats:
                    tokens = feats['x_prenorm']
            else:
                tokens = feats
        if tokens is None:
            tokens = self.vit(x)

        if isinstance(tokens, (list, tuple)):
            tokens = tokens[0]
        if not isinstance(tokens, torch.Tensor):
            raise RuntimeError(f'Unexpected DINO output type: {type(tokens)}')
        if tokens.ndim != 3:
            raise RuntimeError(f'Unexpected DINO token shape: {tuple(tokens.shape)}')

        if tokens.shape[1] == expected_tokens + 1:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != expected_tokens:
            raise RuntimeError(
                f'Unexpected number of DINO tokens: got {tokens.shape[1]}, expected {expected_tokens} '
                f'for img_hw={tuple(x.shape[-2:])}'
            )

        feat = tokens.transpose(1, 2).reshape(B, self.out_dim, Hp, Wp).contiguous()
        return feat

    def forward(self, x):
        feat = self._extract_feature_map(x)
        feat = self.proj(feat)
        return feat

class MultiCamBEVv4DINOv2Clean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 backbone_out_dim: int = 768,
                 backbone_proj_dim: int = 128):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _DINOv2ViTBackbone(
            hub_repo=hub_repo,
            backbone_name=backbone_name,
            out_dim=backbone_out_dim,
            proj_dim=backbone_proj_dim,
            patch_size=14,
        )
        if freeze_backbone:
            for p in self.backbone.vit.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(backbone_proj_dim, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
from src.eval import evaluate_iou
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch, streaming_threshold_sweep
from src.utils.training import cleanup_cuda



In [10]:
cfg['batch_size']

4

In [11]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = DataLoader(
    ds_val,
    batch_size=cfg['val_batch_size'],
    shuffle=False,
    num_workers=cfg['val_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

sample = ds_train_warm[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)



loader_warm batch_size: 4 | workers: 4
loader_train batch_size: 4 | workers: 4
loader_val batch_size: 1 | workers: 0
images (4, 3, 392, 700) torch.float32
intrinsics (4, 3, 3) torch.float32
car2cams (4, 4, 4) torch.float32
rover_id () torch.int64
info_idx <class 'int'> 0
gt (1, 188, 126) torch.int64


In [ ]:
from src.utils.training import update_ema

model = MultiCamBEVv4DINOv2Clean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    hub_repo=cfg['hub_repo'],
    backbone_name=cfg['backbone_name'],
    backbone_out_dim=cfg['backbone_out_dim'],
    backbone_proj_dim=cfg['backbone_proj_dim'],
).to(device)

def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.vit.parameters():
        p.requires_grad = trainable

set_backbone_trainable(model, False)

backbone_params = []
embed_params = []
other_params = []
for name, p in model.named_parameters():
    if name.startswith('backbone.vit.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad = False

print('params M:', sum(p.numel() for p in model.parameters()) / 1e6)
print('backbone frozen at start:', not any(p.requires_grad for p in model.backbone.vit.parameters()))

with torch.no_grad():
    batch = next(iter(loader_warm))
    images = batch['images'].to(device)
    intr = batch['intrinsics'].to(device)
    c2c = batch['car2cams'].to(device)
    rover_id = batch['rover_id'].to(device)

    feat = model.backbone(images.reshape(-1, images.shape[2], images.shape[3], images.shape[4]))
    out = model(images, intr, c2c, rover_id)

    patch_h = cfg['img_hw'][0] // 14
    patch_w = cfg['img_hw'][1] // 14
    print('expected patch grid:', (patch_h, patch_w))
    print('backbone feature shape:', tuple(feat.shape))
    print('sanity output shape:', tuple(out.shape))

cleanup_cuda()


In [13]:
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()
backbone_unfrozen = False

for epoch in range(cfg['epochs']):
    if (not backbone_unfrozen) and epoch >= cfg['freeze_backbone_epochs']:
        set_backbone_trainable(model, True)
        backbone_unfrozen = True
        print(f'unfroze DINO backbone at epoch {epoch:02d}')

    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()

    cleanup_cuda()
    print('start val model @0.5')
    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

    cleanup_cuda()
    print('start val ema @0.5')
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
    cleanup_cuda()

    elapsed = (time.time() - start) / 60
    backbone_grad_enabled = any(p.requires_grad for p in model.backbone.vit.parameters())

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'backbone_trainable': bool(backbone_grad_enabled),
        'minutes': float(elapsed),
    }

    print(
        f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
        f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
        f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
    )

    if val_iou_05 > best_iou:
        best_iou = val_iou_05
        torch.save({
            'model_type': 'v4_dinov2_cleaned',
            'model': model.state_dict(),
            'epoch': epoch,
            'best_iou': float(val_iou_05),
            'best_t': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'best.pt')
        print('  new best model:', val_iou_05)

    if val_iou_05_ema > best_ema_iou:
        best_ema_iou = val_iou_05_ema
        torch.save({
            'model_type': 'v4_dinov2_cleaned',
            'model': model.state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'best_ema_iou': float(val_iou_05_ema),
            'best_t_ema': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'ema_best.pt')
        print('  new best ema:', val_iou_05_ema)

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
    torch.save({
        'model_type': 'v4_dinov2_cleaned',
        'model': model.state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')




ep00 [WARM]: 100%|██████████| 1068/1068 [09:12<00:00,  1.93it/s, bce=1.14, loss=0.839]


start val model @0.5


start val ema @0.5


ep00 | loss 0.921 | val@0.5 0.5174 | ema@0.5 0.4744 | backbone_trainable=False | 11.3m
  new best model: 0.5173848520540572
  new best ema: 0.4743707898028749


ep01 [WARM]: 100%|██████████| 1068/1068 [09:25<00:00,  1.89it/s, bce=0.91, loss=0.855]


...

ep18 | loss 0.505 | val@0.5 0.5960 | ema@0.5 0.5978 | backbone_trainable=True | 537.2m


ep19 [AUG]: 100%|██████████| 1068/1068 [28:30<00:00,  1.60s/it, bce=0.56, loss=0.516]


start val model @0.5


start val ema @0.5


ep19 | loss 0.499 | val@0.5 0.5978 | ema@0.5 0.5973 | backbone_trainable=True | 567.5m


In [14]:
print(1)

1


## Notes

- `clean_merged_info(...)` only writes caches and reports into `RUN_DIR / preproc_cache`.
- The original dataset folders are never modified.
- The `rover_vocab` is built strictly from the cleaned training split, so rare rovers and unseen rovers map to `Other=0`.
- Validation is selected from the merged cleaned pool and targets about 200 samples while staying group-aware by `(rover, ride_date)`.
- DINOv2 weights are expected to come from `torch.hub`, and the notebook will fail early on the sanity-check cell if the backbone cannot be loaded.

